In [3]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import defaultdict

# === CONFIG ===
RATING_CSV_PATH = "ratings.csv"  # your CSV file
POSITIVE_THRESHOLD = 1.0
NEGATIVE_THRESHOLD = 3.0
MIN_COMMON_USERS = 2  # optional filter for reliability

# === STEP 1: LOAD DATA ===
df = pd.read_csv(RATING_CSV_PATH)

# Ensure correct column names
df.columns = ['userId', 'movieId', 'rating']

# === STEP 2: GROUP BY USER TO GET MOVIE PAIRS THEY RATED ===
user_groups = df.groupby('userId')

pair_diff_dict = defaultdict(list)

# For each user, find all movie pairs they rated
for user, group in user_groups:
    rated = group[['movieId', 'rating']].values
    for (m1, r1), (m2, r2) in combinations(rated, 2):
        if m1 == m2:
            continue
        # Ensure consistent movie ordering
        key = tuple(sorted((m1, m2)))
        pair_diff_dict[key].append(abs(r1 - r2))

# === STEP 3: BUILD SIGNED MOVIE–MOVIE GRAPH ===
G = nx.Graph()

for (m1, m2), diffs in pair_diff_dict.items():
    if len(diffs) < MIN_COMMON_USERS:
        continue  # skip weak pairs

    avg_diff = sum(diffs) / len(diffs)

    if avg_diff <= POSITIVE_THRESHOLD:
        G.add_edge(m1, m2, sign=+1, weight=avg_diff)
    elif avg_diff >= NEGATIVE_THRESHOLD:
        G.add_edge(m1, m2, sign=-1, weight=avg_diff)

print(f"Graph built with {G.number_of_nodes()} movies and {G.number_of_edges()} signed edges.")

# === OPTIONAL: Export to visualize or use in GNNs ===
# nx.write_edgelist(G, "signed_movie_graph.edgelist", data=["sign", "weight"])


Graph built with 6268 movies and 2981814 signed edges.


In [4]:
# === STEP 4: EXPORT TO CSV ===
output_edges = []

for u, v, data in G.edges(data=True):
    output_edges.append({
        'movie1': u,
        'movie2': v,
        'sign': data['sign'],
        'weight': data['weight']
    })

# Convert to DataFrame and save
output_df = pd.DataFrame(output_edges)
output_df.to_csv("signed_movie_projection.csv", index=False)

print("Projection saved to 'signed_movie_projection.csv'")


Projection saved to 'signed_movie_projection.csv'


In [5]:
import pandas as pd
from collections import defaultdict

# === FILES ===
EDGES_CSV = 'signed_movie_projection.csv'
MOVIE_INFO_CSV = 'updated_movies_with_year.csv'

# === LOAD CSVs ===
edges_df = pd.read_csv(EDGES_CSV)
meta_df = pd.read_csv(MOVIE_INFO_CSV)

# Ensure consistent column names
meta_df = meta_df[['movieId', 'genres']]

movie_to_genre = dict(zip(meta_df['movieId'], meta_df['genres']))

# === STATS CONTAINERS ===
same_genre_stats = defaultdict(lambda: {'pos': 0, 'neg': 0})
cross_genre_stats = {'pos': 0, 'neg': 0}

# === PROCESS EACH EDGE ===
for _, row in edges_df.iterrows():
    m1, m2, sign = row['movie1'], row['movie2'], row['sign']
    g1 = movie_to_genre.get(m1, None)
    g2 = movie_to_genre.get(m2, None)

    if g1 is None or g2 is None:
        continue  # skip unknown genres

    if g1 == g2:
        if sign == 1:
            same_genre_stats[g1]['pos'] += 1
        elif sign == -1:
            same_genre_stats[g1]['neg'] += 1
    else:
        if sign == 1:
            cross_genre_stats['pos'] += 1
        elif sign == -1:
            cross_genre_stats['neg'] += 1

# === PRINT INSIGHTS ===
print("\n🔍 Same Genre Edge Summary:")
for genre, counts in same_genre_stats.items():
    total = counts['pos'] + counts['neg']
    print(f"- {genre}: {counts['pos']} positive, {counts['neg']} negative, total={total}")

print("\n🔀 Cross-Genre Edge Summary:")
print(f"- Positive edges: {cross_genre_stats['pos']}")
print(f"- Negative edges: {cross_genre_stats['neg']}")



🔍 Same Genre Edge Summary:
- Adventure|Animation|Children|Comedy|Fantasy: 17 positive, 0 negative, total=17
- Comedy|Romance: 5936 positive, 25 negative, total=5961
- Action|Crime|Thriller: 463 positive, 7 negative, total=470
- Mystery|Thriller: 36 positive, 0 negative, total=36
- Crime|Mystery|Thriller: 22 positive, 0 negative, total=22
- Action|Comedy|Horror|Thriller: 1 positive, 0 negative, total=1
- Action|Drama|War: 186 positive, 0 negative, total=186
- Action|Drama|Romance|War: 6 positive, 0 negative, total=6
- Comedy: 23621 positive, 172 negative, total=23793
- Comedy|Drama: 5192 positive, 15 negative, total=5207
- Action|Adventure|Sci-Fi: 596 positive, 2 negative, total=598
- Comedy|Crime|Drama|Thriller: 11 positive, 0 negative, total=11
- Action|Crime|Drama|Thriller: 277 positive, 0 negative, total=277
- Comedy|Drama|Romance|War: 2 positive, 0 negative, total=2
- Action|Thriller: 288 positive, 1 negative, total=289
- Thriller: 155 positive, 1 negative, total=156
- Action|Adve

In [6]:
import pandas as pd
from collections import defaultdict

# === FILES ===
#EDGES_CSV = 'signed_movie_projection.csv'      # movie1, movie2, sign
#META_CSV = 'movie_metadata.csv'                # movie_id, genre, year

# === LOAD DATA ===
#edges_df = pd.read_csv(EDGES_CSV)
#meta_df = pd.read_csv(META_CSV)

# Normalize column names
edges_df = pd.read_csv(EDGES_CSV)
meta_df = pd.read_csv(MOVIE_INFO_CSV)

meta_df = meta_df[['movieId', 'genres', 'year']]

# Helper: Get decade
def get_decade(year):
    try:
        return int(year) // 10 * 10
    except:
        return None

meta_df['decade'] = meta_df['year'].apply(get_decade)

# Mapping: movie_id → (genre, decade)
movie_info = meta_df.set_index('movieId')[['genres', 'decade']].to_dict(orient='index')

# === ANALYSIS DICT ===
group_stats = defaultdict(lambda: {'pos': 0, 'neg': 0})

# === PROCESS EDGES ===
for _, row in edges_df.iterrows():
    m1, m2, sign = row['movie1'], row['movie2'], row['sign']
    info1 = movie_info.get(m1)
    info2 = movie_info.get(m2)

    if not info1 or not info2:
        continue

    # Only analyze edges where both movies share same (genre, decade)
    if info1['genres'] == info2['genres'] and info1['decade'] == info2['decade']:
        key = (info1['genres'], info1['decade'])
        if sign == 1:
            group_stats[key]['pos'] += 1
        elif sign == -1:
            group_stats[key]['neg'] += 1

# === OUTPUT INSIGHTS ===
print("\n🔍 Genre + Decade Consistency Summary:\n")
for (genre, decade), counts in sorted(group_stats.items(), key=lambda x: (x[0][0], x[0][1])):
    total = counts['pos'] + counts['neg']
    if total == 0:
        continue
    ratio = counts['pos'] / total
    print(f"{genre} ({decade}s): +{counts['pos']}, -{counts['neg']} → {ratio:.2%} positive")



🔍 Genre + Decade Consistency Summary:

Action (1980.0s): +2, -0 → 100.00% positive
Action (1990.0s): +5, -0 → 100.00% positive
Action (2000.0s): +3, -0 → 100.00% positive
Action|Adventure (1980.0s): +3, -0 → 100.00% positive
Action|Adventure (1990.0s): +10, -0 → 100.00% positive
Action|Adventure (2000.0s): +5, -0 → 100.00% positive
Action|Adventure (2010.0s): +4, -0 → 100.00% positive
Action|Adventure|Animation (1990.0s): +1, -0 → 100.00% positive
Action|Adventure|Animation (2010.0s): +1, -0 → 100.00% positive
Action|Adventure|Animation|Children|Comedy (2000.0s): +1, -0 → 100.00% positive
Action|Adventure|Animation|Fantasy|Sci-Fi (2000.0s): +1, -0 → 100.00% positive
Action|Adventure|Children|Fantasy (2000.0s): +2, -0 → 100.00% positive
Action|Adventure|Comedy (1990.0s): +2, -0 → 100.00% positive
Action|Adventure|Comedy (2000.0s): +6, -0 → 100.00% positive
Action|Adventure|Comedy (2010.0s): +2, -0 → 100.00% positive
Action|Adventure|Comedy|Crime (1990.0s): +1, -0 → 100.00% positive
Act

In [ ]:
import pandas as pd
from collections import defaultdict

# === FILES ===
#EDGES_CSV = 'signed_movie_projection.csv'      # movie1, movie2, sign
#META_CSV = 'movie_metadata.csv'                # movie_id, genre, year

# === PARAMETERS ===
TOP_N = 10  # number of groups to display
SORT_BY = 'positive_ratio'  # or 'total_edges'

# === LOAD DATA ===
#edges_df = pd.read_csv(EDGES_CSV)
#meta_df = pd.read_csv(META_CSV)

# Normalize column names
meta_df = meta_df[['movieId', 'genres', 'year']]

# 5-year binning
def get_5year_bin(year):
    try:
        return int(year) // 5 * 5
    except:
        return None

meta_df['time_bin'] = meta_df['year'].apply(get_5year_bin)

# Movie metadata lookup
movie_info = meta_df.set_index('movieId')[['genres', 'time_bin']].to_dict(orient='index')

# === COLLECT STATS ===
group_stats = []

for _, row in edges_df.iterrows():
    m1, m2, sign = row['movie1'], row['movie2'], row['sign']
    info1 = movie_info.get(m1)
    info2 = movie_info.get(m2)

    if not info1 or not info2:
        continue

    if info1['genres'] == info2['genres'] and info1['time_bin'] == info2['time_bin']:
        key = (info1['genres'], info1['time_bin'])
        found = next((g for g in group_stats if g['key'] == key), None)
        if not found:
            found = {'key': key, 'pos': 0, 'neg': 0}
            group_stats.append(found)
        if sign == 1:
            found['pos'] += 1
        elif sign == -1:
            found['neg'] += 1

# === ENRICH WITH METRICS ===
for entry in group_stats:
    total = entry['pos'] + entry['neg']
    entry['total'] = total
    entry['positive_ratio'] = entry['pos'] / total if total > 0 else 0.0

# === SORT + DISPLAY TOP N ===
sorted_stats = sorted(group_stats, key=lambda x: x[SORT_BY], reverse=True)[:TOP_N]

print(f"\n Top {TOP_N} (sorted by {SORT_BY}):\n")
for entry in sorted_stats:
    genre, bin_start = entry['key']
    print(f"{genre} ({bin_start}–{bin_start+4}): +{entry['pos']}, -{entry['neg']}, "
          f"Total={entry['total']}, Positive %={entry['positive_ratio']:.2%}")



 Top 10 (sorted by positive_ratio):

Adventure|Animation|Children|Comedy|Fantasy (1995.0–1999.0): +3, -0, Total=3, Positive %=100.00%
Comedy|Romance (1995.0–1999.0): +354, -0, Total=354, Positive %=100.00%
Action|Crime|Thriller (1995.0–1999.0): +18, -0, Total=18, Positive %=100.00%
Mystery|Thriller (1995.0–1999.0): +2, -0, Total=2, Positive %=100.00%
Crime|Mystery|Thriller (1995.0–1999.0): +1, -0, Total=1, Positive %=100.00%
Action|Drama|War (1995.0–1999.0): +1, -0, Total=1, Positive %=100.00%
Comedy|Drama (1990.0–1994.0): +137, -0, Total=137, Positive %=100.00%
Comedy|Crime|Drama|Thriller (1990.0–1994.0): +1, -0, Total=1, Positive %=100.00%
Action|Crime|Drama|Thriller (1990.0–1994.0): +2, -0, Total=2, Positive %=100.00%
Action|Thriller (1990.0–1994.0): +4, -0, Total=4, Positive %=100.00%


In [ ]:
import pandas as pd
from collections import defaultdict

# === FILE PATHS ===
#EDGES_CSV = 'signed_movie_projection.csv'      # columns: movie1, movie2, sign
#META_CSV = 'movie_metadata.csv'                # columns: movie_id, genre, year

# === CONFIG ===
TOP_N = 10
SORT_BY = 'total_edges'         # options: 'positive_ratio', 'total_edges'
MIN_EDGES = 20                  # minimum edges in a group to be considered

# === LOAD DATA ===
#edges_df = pd.read_csv(EDGES_CSV)
#meta_df = pd.read_csv(META_CSV)
meta_df = meta_df[['movieId', 'genres', 'year']]

# === Time bucketing (5-year bins) ===
def get_5year_bin(year):
    try:
        return int(year) // 5 * 5
    except:
        return None

meta_df['time_bin'] = meta_df['year'].apply(get_5year_bin)

# Mapping: movie_id → (genre, 5yr_bin)
movie_info = meta_df.set_index('movieId')[['genres', 'time_bin']].to_dict(orient='index')

# === COLLECT STATS ===
group_stats = []

for _, row in edges_df.iterrows():
    m1, m2, sign = row['movie1'], row['movie2'], row['sign']
    info1 = movie_info.get(m1)
    info2 = movie_info.get(m2)

    if not info1 or not info2:
        continue

    if info1['genres'] == info2['genres'] and info1['time_bin'] == info2['time_bin']:
        key = (info1['genres'], info1['time_bin'])
        found = next((g for g in group_stats if g['key'] == key), None)
        if not found:
            found = {'key': key, 'pos': 0, 'neg': 0}
            group_stats.append(found)
        if sign == 1:
            found['pos'] += 1
        elif sign == -1:
            found['neg'] += 1

# === ENRICH + FILTER ===
filtered_stats = []
for entry in group_stats:
    total = entry['pos'] + entry['neg']
    if total < MIN_EDGES:
        continue
    entry['total_edges'] = total
    entry['positive_ratio'] = entry['pos'] / total
    filtered_stats.append(entry)

# === SORT + DISPLAY ===
sorted_stats = sorted(filtered_stats, key=lambda x: x[SORT_BY], reverse=True)[:TOP_N]

print(f"\n top {TOP_N} (sorted by {SORT_BY}, min {MIN_EDGES} edges):\n")
for entry in sorted_stats:
    genre, bin_start = entry['key']
    print(f"{genre} ({bin_start}–{bin_start+4}): "
          f"+{entry['pos']}, -{entry['neg']}, "
          f"Total={entry['total_edges']}, "
          f"Positive %={entry['positive_ratio']:.2%}")



🔝 Top 10 (sorted by total_edges, min 20 edges):

Drama (2000.0–2004.0): +1498, -0, Total=1498, Positive %=100.00%
Comedy (1995.0–1999.0): +1487, -7, Total=1494, Positive %=99.53%
Comedy (2000.0–2004.0): +1102, -1, Total=1103, Positive %=99.91%
Drama (1995.0–1999.0): +905, -1, Total=906, Positive %=99.89%
Drama (1990.0–1994.0): +667, -2, Total=669, Positive %=99.70%
Comedy (1985.0–1989.0): +510, -4, Total=514, Positive %=99.22%
Comedy|Romance (2000.0–2004.0): +510, -0, Total=510, Positive %=100.00%
Comedy (1990.0–1994.0): +483, -3, Total=486, Positive %=99.38%
Comedy (2010.0–2014.0): +429, -3, Total=432, Positive %=99.31%
Drama|Romance (1995.0–1999.0): +389, -5, Total=394, Positive %=98.73%


In [9]:
import pandas as pd
import networkx as nx
from cdlib import algorithms
from cdlib import evaluation

# === Load edge list (signed projection) ===
           # columns: movie_id, genre, year
meta_df = meta_df[['movieId', 'genres', 'year']]


# === Build Signed Graph ===
G = nx.Graph()

for _, row in edges_df.iterrows():
    m1, m2, sign = row['movie1'], row['movie2'], row['sign']
    G.add_edge(m1, m2, weight=sign)

# === Run Signed Louvain ===
# This uses both positive and negative edge weights



Note: to be able to use all crisp methods, you need to install some additional packages:  {'bayanpy', 'graph_tool', 'leidenalg', 'infomap'}
Note: to be able to use all crisp methods, you need to install some additional packages:  {'ASLPAw', 'pyclustering'}
Note: to be able to use all crisp methods, you need to install some additional packages:  {'leidenalg', 'infomap'}


In [12]:
# Create subgraphs
G_pos = nx.Graph((u, v, d) for u, v, d in G.edges(data=True) if d["weight"] > 0)
G_neg = nx.Graph((u, v, d) for u, v, d in G.edges(data=True) if d["weight"] < 0)

# Detect communities separately
pos_communities = algorithms.louvain(G_pos, weight="weight")
#neg_communities = algorithms.louvain(G_neg, weight="weight")


In [11]:
communities = algorithms.louvain(G, weight="weight")

# === Output: Community Summary ===
print(f"🔍 Detected {len(communities.communities)} communities.\n")

# Optional: Print first 5 communities
for i, comm in enumerate(communities.communities[:5]):
    print(f"Community {i+1} ({len(comm)} movies): {comm[:10]}{'...' if len(comm) > 10 else ''}")

ValueError: Bad node degree (-11.0)

In [ ]:
import igraph as ig
from cdlib import algorithms
edges = [(u, v, float(d['weight'])) for u, v, d in G.edges(data=True)]

# Build iGraph object with float weights
g_igraph = ig.Graph.TupleList(edges, edge_attrs=["weight"])

# Run Leiden (only works with unsigned unless manually extended)
communities = algorithms.leiden(g_igraph, weights="weight")


# === Output: Community Summary ===
print(f"🔍 Detected {len(communities.communities)} communities.\n")

# Optional: Print first 5 communities
for i, comm in enumerate(communities.communities[:5]):
    print(f"Community {i+1} ({len(comm)} movies): {comm[:10]}{'...' if len(comm) > 10 else ''}")

BaseException: Could not construct partition: Cannot accept negative weights.

In [14]:
def year_bin(year):
    try:
        return int(year) // 5 * 5
    except:
        return None

meta_df["year_bin"] = meta_df["year"].apply(year_bin)
# === Mappings for genre and year bin ===
genre_map = dict(zip(meta_df['movieId'], meta_df['genres']))
year_map = dict(zip(meta_df['movieId'], meta_df['year_bin']))

# === Analyze Communities ===
for i, comm in enumerate(pos_communities.communities[:5]):  # Only first 5 communities
    genre_counts = {}
    year_counts = {}

    for movie in comm:
        genre = genre_map.get(movie)
        year = year_map.get(movie)

        if genre:
            genre_counts[genre] = genre_counts.get(genre, 0) + 1
        if year:
            year_counts[year] = year_counts.get(year, 0) + 1

    dominant_genre = max(genre_counts, key=genre_counts.get) if genre_counts else "Unknown"
    dominant_year = max(year_counts, key=year_counts.get) if year_counts else "Unknown"

    print(f"🧩 Community {i+1}:")
    print(f"   Size             : {len(comm)} movies")
    print(f"   Dominant Genre   : {dominant_genre} ({genre_counts.get(dominant_genre, 0)})")
    print(f"   Dominant 5yr Bin : {dominant_year}–{int(dominant_year)+4 if dominant_year != 'Unknown' else ''} ({year_counts.get(dominant_year, 0)})\n")

🧩 Community 1:
   Size             : 2380 movies
   Dominant Genre   : Drama (427)
   Dominant 5yr Bin : 1995.0–1999 (406)

🧩 Community 2:
   Size             : 2045 movies
   Dominant Genre   : Comedy (298)
   Dominant 5yr Bin : 1995.0–1999 (542)

🧩 Community 3:
   Size             : 1837 movies
   Dominant Genre   : Comedy (167)
   Dominant 5yr Bin : 2010.0–2014 (638)



In [15]:
import pandas as pd
import networkx as nx
from collections import defaultdict

# === Load Data ===
#edges_df = pd.read_csv("signed_movie_projection.csv")  # movie1, movie2, sign
#meta_df = pd.read_csv("movie_metadata.csv")            # movie_id, genre, year
user_ratings_df = pd.read_csv("ratings.csv")  # user_id, movie_id, rating

#meta_df.columns = ['movie_id', 'genre', 'year']

# === Construct the Signed Graph ===
#G = nx.Graph()
#for _, row in edges_df.iterrows():
#    G.add_edge(row['movie1'], row['movie2'], weight=row['sign'])

# === Genre Mapping ===
#genre_map = dict(zip(meta_df['movie_id'], meta_df['genre']))

# === Get Movies Liked by a Specific User ===
def get_user_likes(user_id, ratings_df):
    # Get movies liked by the user (rating > 0)
    user_likes = ratings_df[ratings_df['rating'] > 3]
    return user_likes[user_likes['userId'] == user_id]['movieId'].tolist()

# === Genre-Aware Recommendation Logic ===
def genre_aware_recommendations(user_id, ratings_df, G, genre_map, top_n=5):
    user_movies = [m for m in get_user_likes(user_id, ratings_df) if m in G]
    recommendations = defaultdict(int)  # Movie to score

    # List of movies the user already liked
    liked_movies = set(user_movies)

    # Loop through each liked movie and find its positive neighbors
    for movie in liked_movies:

        neighbors = [
            nbr for nbr in G.neighbors(movie)
            if G[movie][nbr]['weight'] > 0
        ]


        for neighbor in neighbors:
            # Recommend based on genre similarity
            if genre_map.get(movie) == genre_map.get(neighbor):  # Same genre
                recommendations[neighbor] += 1
            else:  # Different genre, can still recommend
                recommendations[neighbor] += 0.5

    # Sort movies by score (higher score = more similar)
    sorted_recommendations = sorted(recommendations.items(), key=lambda x: x[1], reverse=True)

    # Return top N recommendations
    return sorted_recommendations[:top_n]

# === Example User Interaction ===
user_id = 123  # Example user ID
top_recommendations = genre_aware_recommendations(user_id, user_ratings_df, G, genre_map, top_n=10)

print(f"\n🔝 Genre-Aware Recommendations for User {user_id}:")
for movie, score in top_recommendations:
    movie_genre = genre_map.get(movie, "Unknown")
    print(f"{movie} (Genre: {movie_genre}), Score: {score:.2f}")



🔝 Genre-Aware Recommendations for User 123:
51540.0 (Genre: Crime|Drama|Thriller), Score: 29.50
80489.0 (Genre: Crime|Drama|Thriller), Score: 29.00
1961.0 (Genre: Drama), Score: 29.00
4776.0 (Genre: Crime|Drama|Thriller), Score: 29.00
44199.0 (Genre: Crime|Drama|Thriller), Score: 29.00
5989.0 (Genre: Crime|Drama), Score: 28.50
8958.0 (Genre: Drama), Score: 28.50
81845.0 (Genre: Drama), Score: 28.50
55765.0 (Genre: Crime|Drama|Thriller), Score: 28.50
115569.0 (Genre: Crime|Drama|Thriller), Score: 28.50


In [8]:
!pip install cdlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.6/263.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 115.6 MB/s eta 0:00:00


In [ ]:
!pip install bayanpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.0/122.0 kB 9.0 MB/s eta 0:00:00


In [ ]:
!pip install leidenalg igraph


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 18.1 MB/s eta 0:00:00
